In [2]:
import numpy as np

import jax.numpy as jnp
from jax import jit, vmap, grad
import jax.random

import scipy.optimize
import matplotlib.pyplot as plt

import emcee
import corner
# import tensorflow_probability.substrates.jax as tfp

import numpyro.distributions as dist
import numpyro
from numpyro import infer

# Exercise 1

For the linear model $y_i \sim N(mx_i + b, \sigma_i)$ and data from `../lectures/data/linear_fits/data_0.txt` do the following:

1. Analytically calculate the FIM using either the Hessian definition or the grad definition.
2. Write a function that calculates this based on the $x$-values and $\sigma$.
3. Implement the likelihood in JAX and use `jax.hessian` or `jax.grad` to calculate the FIM and compare to (2). If you use `jax.hessian`, check that properties of the analytic FIM propagate here (e.g. whether it is independent of certain variables).

In [3]:
x, y, y_err = np.loadtxt("../lectures/data/linear_fits/data_0.txt", unpack=True)



# Exercise 2: Hierarchical models and probabilistic programming

We have test scores $y_{ij}$ for schools across several cities, where $j$ labels the city and $i$ labels the school within that city.
Schools in the same city share a mean test score $\theta_j$, and the city means are themselves drawn from a country-wide Gaussian:

$$y_{ij} \sim \mathcal N_y(\theta_j, \sigma_{ij}^2)$$

$$\theta_j \sim \mathcal N_\theta(\mu, \tau^2)$$
$$\mu \sim \mathcal U(0, 100)$$
$$\log_{10} \tau \sim \mathcal U(-2, 2)$$

where we have used a Jeffreys prior on $\tau$.

The data are in `lectures/data/fake_school_data.txt`: 20 cities with 10 schools each.
The file has 40 columns that alternate between the scores of the schools in a city and the variances $\sigma_{ij}^2$ of those scores
(column 1: scores in city 1, column 2: variances in city 1, column 3: scores in city 2, ...).

1. Implement this model in `numpyro`.
2. Sample the posterior of $\mu$, $\tau$ and the $\theta_j$ with NUTS.
3. Compare the posterior means of the $\theta_j$ with the per-city average scores. What does the hierarchical model do to them?

In [36]:
# Notation:
#   n_city             number of cities (groups), here 20
#   n_school_per_city  number of schools per city, here 10
school_table = np.loadtxt("../lectures/data/fake_school_data.txt")  # (n_school_per_city, 2 * n_city)
print(school_table.shape)

# Columns alternate (scores, variances) per city; transpose so cities are rows
scores = school_table[:, 0::2].T           # (n_city, n_school_per_city)
score_variances = school_table[:, 1::2].T  # (n_city, n_school_per_city)
score_sigmas = np.sqrt(score_variances)    # (n_city, n_school_per_city)

n_city, n_school_per_city = scores.shape
print(f"{n_city} cities with {n_school_per_city} schools each")

num_samples=10_000

(10, 40)
20 cities with 10 schools each


In [23]:
def model(scores, scores_sigmas):
    # priors
    mu = numpyro.sample("mu", dist.Uniform(low=0, high=100))
    log_tau = numpyro.sample("log_tau", dist.Uniform(low=-2, high=2))
    theta_j = numpyro.sample("theta", dist.TruncatedNormal(loc=mu, scale=10**log_tau, low=0, high=100).expand([n_city]))
    
    
    # likelihood
    numpyro.sample("obs", dist.Normal(loc=theta_j[:,None], scale=scores_sigmas), obs=scores)


master_key = jax.random.PRNGKey(42)
rng_key, master_key = jax.random.split(master_key)



kernel = numpyro.infer.NUTS(model)
mcmc = numpyro.infer.MCMC(kernel, num_warmup=1000, num_samples=num_samples)
mcmc.run(
    rng_key, scores=scores, scores_sigmas=score_sigmas,
)

    


sample: 100%|██████████| 3000/3000 [00:01<00:00, 1516.85it/s, 7 steps of size 5.28e-01. acc. prob=0.88]


In [24]:
mcmc.print_summary()
samples = mcmc.get_samples()


                mean       std    median      5.0%     95.0%     n_eff     r_hat
   log_tau      0.88      0.08      0.87      0.75      1.01   1862.03      1.00
        mu     69.29      1.84     69.28     66.30     72.26   2940.63      1.00
  theta[0]     62.53      1.97     62.54     58.96     65.42   4010.62      1.00
  theta[1]     67.39      2.34     67.38     63.54     71.27   3749.79      1.00
  theta[2]     55.90      2.23     55.91     52.33     59.65   4977.64      1.00
  theta[3]     86.58      2.22     86.61     82.61     89.99   4266.34      1.00
  theta[4]     76.10      2.03     76.12     72.69     79.36   4799.24      1.00
  theta[5]     65.00      2.20     64.96     61.31     68.55   4238.26      1.00
  theta[6]     68.21      1.98     68.22     65.16     71.59   3977.00      1.00
  theta[7]     75.02      2.51     75.01     70.72     78.95   4068.31      1.00
  theta[8]     68.07      2.10     68.09     64.59     71.37   3895.65      1.00
  theta[9]     71.96      1

In [37]:
posterior_predictive = infer.Predictive(model=model, posterior_samples=dict(mu=samples["mu"], log_tau=samples["log_tau"]))